In [1]:
# 一鍵在 Renku 環境中強行安裝表格提取武器
!pip install pdfplumber tabulate --quiet
print("📦 pdfplumber 與 tabulate 工具箱安裝完成！")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
📦 pdfplumber 與 tabulate 工具箱安裝完成！


In [5]:
import os
import pandas as pd
import pdfplumber

# =========================================================================
# 🏆 【RAG 表格清洗優化模組】輕量級快顯焊接器 (Lightweight Table Extractor)
# =========================================================================

def extract_and_stitch_pdf_tables(pdf_path):
    if not os.path.exists(pdf_path):
        print(f"   ❌ 找不到目標檔案: {pdf_path}")
        return []
        
    pdf_name = os.path.basename(pdf_path)
    table_chunks = []
    
    last_known_header = None
    last_known_col_count = 0
    
    print(f"📖 正在載入 PDF 檔案: {pdf_name}...")
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            print(f"   🎯 成功開啟！總計 {total_pages} 頁。開始逐頁極速檢索表格...")
            
            for page_num in range(total_pages):
                page = pdf.pages[page_num]
                
                # 【優化核心】改用輕量級的 find_tables，防範複雜 PDF 線條導致的 Kernel 崩潰
                table_objects = page.find_tables()
                if not table_objects:
                    continue
                
                # 每當在特定頁面撈到表格，立刻噴出日誌，讓我們知道程式還活著！
                print(f"   ⚡ 偵測到第 {page_num + 1} 頁內含結構化表格，正在進行語義格式化...")
                
                for t_idx, table_obj in enumerate(table_objects):
                    table = table_obj.extract()
                    if not table:
                        continue
                        
                    clean_table = [[(cell.strip() if cell else "") for cell in row] for row in table if any(row)]
                    if not clean_table:
                        continue
                        
                    current_col_count = len(clean_table[0])
                    is_stitched = False
                    
                    # 跨頁斷裂表格幾何判定邏輯
                    if t_idx == 0 and last_known_header and current_col_count == last_known_col_count:
                        if clean_table[0] != last_known_header:
                            clean_table.insert(0, last_known_header)
                            is_stitched = True
                            print(f"      🔗 成功！檢測到跨頁斷裂，已將上一頁表頭焊接至第 {page_num + 1} 頁表格頂端！")
                    
                    try:
                        headers = clean_table[0]
                        data_rows = clean_table[1:]
                        sanitized_rows = [r + [""] * (len(headers) - len(r)) if len(r) < len(headers) else r[:len(headers)] for r in data_rows]
                        df_temp = pd.DataFrame(sanitized_rows, columns=headers)
                        markdown_table = df_temp.to_markdown(index=False)
                    except:
                        markdown_table = "\n".join([" | ".join(row) for row in clean_table])
                    
                    if not is_stitched:
                        last_known_header = clean_table[0]
                        last_known_col_count = current_col_count
                    
                    status_tag = "STITCHED_MULTIDOC_TABLE" if is_stitched else "STANDARD_TABLE"
                    table_content = f"[CRITICAL STRUCTURED TABLE - FROM {pdf_name} PAGE {page_num+1} ({status_tag})]\n{markdown_table}"
                    
                    table_chunks.append({
                        "page_content": table_content,
                        "metadata": {
                            "source_file": pdf_name,
                            "page": page_num + 1,
                            "data_type": "structured_table",
                            "stitched": is_stitched
                        }
                    })
        print(f"   慶賀！{pdf_name} 順利解碼完畢，成功捕獲 {len(table_chunks)} 個表格。")
    except Exception as e:
        print(f"   ❌ 處理 {pdf_name} 時發生異常阻礙: {e}")
        
    return table_chunks

# =========================================================================
# 🎯 一鍵大批量執行：讀取 Sandy 資料夾下的 4 份 PDF
# =========================================================================
pdf_folder_path = "/home/renku/work/Durham-Hackathon-2026-w1t2/Sandy/real_pdf_files"
master_table_pool = []

print("📡 [雷達啟動] 正在掃描實體目標資料夾...")

if not os.path.exists(pdf_folder_path):
    print(f"❌ 錯誤：找不到 Renku 絕對路徑：{pdf_folder_path}")
else:
    pdf_files = [f for f in os.listdir(pdf_folder_path) if f.endswith('.pdf')]
    print(f"📂 成功鎖定資料夾！待處理的 PDF 列表為：{pdf_files}\n")
    
    for pdf_name in pdf_files:
        full_absolute_path = os.path.join(pdf_folder_path, pdf_name)
        master_table_pool.extend(extract_and_stitch_pdf_tables(full_absolute_path))
        print("-" * 50)
        
    print("\n" + "="*60)
    print(f"✅ 全自動清洗結束！共計生成 {len(master_table_pool)} 個標準 Markdown 表格區塊。")
    print("="*60)

📡 [雷達啟動] 正在掃描實體目標資料夾...
📂 成功鎖定資料夾！待處理的 PDF 列表為：['World_Inequality_Report_2026.pdf', 'swissre_sigma-1_2024_english.pdf', 'Web Version _E-Government Survey 2024 11102024.pdf', 'natural-catastrophe-and-climate-report-2023.pdf']

📖 正在載入 PDF 檔案: World_Inequality_Report_2026.pdf...
   🎯 成功開啟！總計 208 頁。開始逐頁極速檢索表格...
   ⚡ 偵測到第 13 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 14 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 15 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 16 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 17 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 18 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 19 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 20 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 21 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 22 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 23 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 24 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 25 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 36 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 37 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 38 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 40 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 42 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 43 頁內含結構化表格，正在進行語義格式化...
   ⚡ 偵測到第 44 頁內含結構化表格，正在進

KeyboardInterrupt: 